In [1]:
import PIL

import os
import base64
import json
from io import BytesIO
from pathlib import Path

import torch

import datasets
from transformers import Qwen3_5ForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel, LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

from utils import load_base_model, load_dataset


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

/Data/joao.giordani-donasolo/gen_ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


In [2]:
with open("../output_images/entities.jsonl", "r") as f:
    entities = f.readlines()

entities = [json.loads(e) for e in entities]
entities[0]

{'id': 'img_005',
 'page': 43,
 'caption': 'Reversible cell injury and necrosis. The principal cellular alterations that characterize reversible cell injury and necrosis are illustrated. By conven- tion, reversible injury is considered to culminate in necrosis if the injurious stimulus is not removed.',
 'summary': 'The image illustrates the progression of cellular injury, specifically showing three stages: a normal cell, a cell undergoing reversible injury, and a cell undergoing necrosis. The normal cell is depicted with intact organelles and a healthy nucleus. The cell in reversible injury shows signs of swelling and organelle damage, such as mitochondrial swelling and ribosome detachment. The necrotic cell exhibits severe damage, including nuclear fragmentation (karyorrhexis), nuclear condensation (pyknosis), and nuclear dissolution (karyolysis), along with cytoplasmic vacuolization and membrane disruption. The caption explains that reversible injury can progress to necrosis if the 

In [3]:
relations = []
for p in Path("../output_images/").glob("relations*"):
    with open(p, "r") as f:
        lines = f.readlines()
    relations.extend([json.loads(l) for l in lines])
relations[0]

{'id': 'img_005',
 'page': 43,
 'caption': 'Reversible cell injury and necrosis. The principal cellular alterations that characterize reversible cell injury and necrosis are illustrated. By conven- tion, reversible injury is considered to culminate in necrosis if the injurious stimulus is not removed.',
 'entities': ['Cell', 'Nucleus'],
 'relation': "### Discussion of <image_1> in relation to Cell\nThe image illustrates a progression of cellular states, beginning with a healthy cell and advancing through stages of reversible injury to necrosis. The cell is depicted as a structured unit with distinct organelles, including the nucleus, mitochondria, and other cytoplasmic components. In the initial stage, the cell appears intact and functional, with a well-defined nucleus and organized internal structures. As the cell undergoes reversible injury, subtle changes become apparent, such as swelling and alterations in the nuclear envelope. If the injurious stimulus persists, the cell progresse

In [4]:
ds = datasets.Dataset.from_list(relations)

In [13]:
ds = datasets.load_from_disk("../relations_dataset")

In [16]:
ds[0]["id"]

'img_005'

In [ ]:
from PIL import Image

def load_image_by_id(example):
    image_path = f"../output_images/{example['id']}.jpg"
    example["image"] = Image.open(image_path).convert("RGB")
    return example

ds = ds.map(load_image_by_id, num_proc=os.cpu_count())

Map:  27%|██▋       | 4901/18248 [00:35<00:37, 360.38 examples/s]